# Fase 1: Línea Base — StreamLoop Churn Prediction

**Objetivo:** Construir un pipeline de preprocesamiento + clasificador con hiperparámetros por defecto como punto de comparación para las fases de ajuste posteriores.

In [3]:
# Importar librerías necesarias
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, cross_val_score, RepeatedKFold
from sklearn.preprocessing import StandardScaler, LabelEncoder, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, recall_score, f1_score, precision_score, roc_auc_score
import warnings
warnings.filterwarnings('ignore')

print('✅ Librerías importadas correctamente')

✅ Librerías importadas correctamente


In [2]:
# Verificar versiones
print(f'NumPy versión: {np.__version__}')
print(f'Pandas versión: {pd.__version__}')
print(f'Scikit-learn versión: {__import__("sklearn").__version__}')

NumPy versión: 2.5.3
Pandas versión: 3.0.6
Scikit-learn versión: 1.9.1


## 1. Carga del dataset

Cargamos los datos directamente desde la URL del dataset público de IBM Telco Customer Churn.

In [4]:
# Cargar dataset
url = 'https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv'
df = pd.read_csv(url)

print(f'Shape: {df.shape}')
print(f'\nColumnas ({len(df.columns)}):')
print(df.columns.tolist())
print(f'\nDistribución de Churn:')
print(df['Churn'].value_counts())
print(f'\nProporción:')
print(df['Churn'].value_counts(normalize=True))
print(f'\nValores nulos: {df.isnull().sum().sum()}')
df.head(3)

Shape: (7043, 21)

Columnas (21):
['customerID', 'gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'MonthlyCharges', 'TotalCharges', 'Churn']

Distribución de Churn:
Churn
No     5174
Yes    1869
Name: count, dtype: int64

Proporción:
Churn
No     0.73463
Yes    0.26537
Name: proportion, dtype: float64

Valores nulos: 0


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes


## 2. Limpieza mínima

- Eliminamos `customerID` (es un identificador único, no predictivo)
- Convertimos `TotalCharges` a numérico (11 valores vacíos → los rellenamos con 0 porque son clientes nuevos con tenure=0)
- Mapeamos `SeniorCitizen` (0/1) a string para tratarlo como categoría binaria
- Codificamos target `Churn` a 0/1

In [5]:
# Limpieza

# 1. Eliminar customerID (no aporta info predictiva)
df.drop('customerID', axis=1, inplace=True)

# 2. Convertir TotalCharges a numérico (los vacíos son clientes con tenure=0)
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce').fillna(0)

# 3. SeniorCitizen es binario (0/1) — lo tratamos como categoría
df['SeniorCitizen'] = df['SeniorCitizen'].astype(str)

# 4. Codificar target: Churn Yes=1, No=0
df['Churn'] = df['Churn'].map({'Yes': 1, 'No': 0})

print(f'Shape después de limpieza: {df.shape}')
print(f'\nTipos actualizados:')
print(df.dtypes)
print(f'\nTotalCharges - valores únicos (primeros 5): {df["TotalCharges"].unique()[:5]}')
print(f'TotalCharges - dtype: {df["TotalCharges"].dtype}')
print(f'\nTarget distribution:')
print(df['Churn'].value_counts())
print(f'Rango de edades (SeniorCitizen): {df["SeniorCitizen"].unique()}')

Shape después de limpieza: (7043, 20)

Tipos actualizados:
gender                  str
SeniorCitizen           str
Partner                 str
Dependents              str
tenure                int64
PhoneService            str
MultipleLines           str
InternetService         str
OnlineSecurity          str
OnlineBackup            str
DeviceProtection        str
TechSupport             str
StreamingTV             str
StreamingMovies         str
Contract                str
PaperlessBilling        str
PaymentMethod           str
MonthlyCharges      float64
TotalCharges        float64
Churn                 int64
dtype: object

TotalCharges - valores únicos (primeros 5): [  29.85 1889.5   108.15 1840.75  151.65]
TotalCharges - dtype: float64

Target distribution:
Churn
0    5174
1    1869
Name: count, dtype: int64
Rango de edades (SeniorCitizen): <StringArray>
['0', '1']
Length: 2, dtype: str


## 3. Separación train/test

**Importante:** El test set se separa AHORA y no se toca hasta el final del proyecto. Así evitamos contaminarlo durante la búsqueda de hiperparámetros.

Usamos **estratificación** para mantener la misma proporción de Churn en train y test.

In [6]:
# Separar features (X) y target (y)
X = df.drop('Churn', axis=1)
y = df['Churn']

# División train/test (80/20) con estratificación
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Train size: {X_train.shape[0]} ({X_train.shape[0]/len(df)*100:.1f}%)')
print(f'Test size:  {X_test.shape[0]} ({X_test.shape[0]/len(df)*100:.1f}%)')
print(f'\nChurn en train:')
print(y_train.value_counts(normalize=True))
print(f'\nChurn en test:')
print(y_test.value_counts(normalize=True))

Train size: 5634 (80.0%)
Test size:  1409 (20.0%)

Churn en train:
Churn
0    0.734647
1    0.265353
Name: proportion, dtype: float64

Churn en test:
Churn
0    0.734564
1    0.265436
Name: proportion, dtype: float64


## 4. Pipeline de preprocesamiento + clasificador

**Regla clave:** Todo el preprocesamiento vive DENTRO del Pipeline. Nunca se aplica fuera, porque:
- El Pipeline aprende las transformaciones SOLO del train set (evita data leakage)
- Se aplica automáticamente al test set y a futuros datos nuevos
- No podemos contaminar el test con estadísticas del train

### Columnas
- **Numéricas:** tenure, MonthlyCharges, TotalCharges → `StandardScaler`
- **Categóricas (binarias/low-cardinality):** gender, SeniorCitizen, Partner, Dependents, PhoneService, MultipleLines, InternetService, OnlineSecurity, OnlineBackup, DeviceProtection, TechSupport, StreamingTV, StreamingMovies, Contract, PaperlessBilling, PaymentMethod → `OneHotEncoder` (drop='first' para evitar multicolinealidad)

**Clasificador base:** `RandomForestClassifier` con hiperparámetros por defecto.

In [7]:
# Identificar tipos de columnas
numeric_features = ['tenure', 'MonthlyCharges', 'TotalCharges']
categorical_features = ['gender', 'SeniorCitizen', 'Partner', 'Dependents', 'PhoneService',
                        'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup',
                        'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies',
                        'Contract', 'PaperlessBilling', 'PaymentMethod']

# Transformadores
numeric_transformer = StandardScaler()
categorical_transformer = OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore')

# ColumnTransformer: une ambos preprocesamientos
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ]
)

# Pipeline completo: preprocesamiento + clasificador
pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(random_state=42))
])

print('✅ Pipeline creado')
print(pipeline)

✅ Pipeline creado
Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num', StandardScaler(),
                                                  ['tenure', 'MonthlyCharges',
                                                   'TotalCharges']),
                                                 ('cat',
                                                  OneHotEncoder(drop='first',
                                                                handle_unknown='ignore',
                                                                sparse_output=False),
                                                  ['gender', 'SeniorCitizen',
                                                   'Partner', 'Dependents',
                                                   'PhoneService',
                                                   'MultipleLines',
                                                   'InternetService',
                                                   'Onlin

## 5. Entrenar línea base (hiperparámetros por defecto)

Entrenamos con los valores por defecto de RandomForest. Este resultado servirá como **punto de comparación** para todo el ajuste posterior.

In [19]:
# Entrenar pipeline con parámetros por defecto
pipeline.fit(X_train, y_train)

print('✅ Pipeline entrenado (línea base)')

# Predicciones en test
y_pred = pipeline.predict(X_test)

# Métricas
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
roc_auc = roc_auc_score(y_test, y_pred)

print('\n🔵 LÍNEA BASE — Desempeño en test set')
print(f'Accuracy:  {accuracy:.4f}')
print(f'Precision: {precision:.4f}')
print(f'Recall:    {recall:.4f}')
print(f'F1-score:  {f1:.4f}')
print(f'ROC AUC:   {roc_auc:.4f}')

print('\nMatriz de confusión:')
print(confusion_matrix(y_test, y_pred))

print('\nClassification Report:')
print(classification_report(y_test, y_pred))

✅ Pipeline entrenado (línea base)

🔵 LÍNEA BASE — Desempeño en test set
Accuracy:  0.7921
Precision: 0.6421
Recall:    0.4893
F1-score:  0.5554
ROC AUC:   0.6954

Matriz de confusión:
[[933 102]
 [191 183]]

Classification Report:
              precision    recall  f1-score   support

           0       0.83      0.90      0.86      1035
           1       0.64      0.49      0.56       374

    accuracy                           0.79      1409
   macro avg       0.74      0.70      0.71      1409
weighted avg       0.78      0.79      0.78      1409



## 6. Validación cruzada en train (estabilidad de la línea base)

Para no confiar en un solo número, corremos validación cruzada en el train set y vemos cuánto varía el Recall entre folds. Esto nos dirá si el modelo es consistentemente bueno o solo tuvo suerte.

In [10]:
# Validación cruzada de 5 folds para comparar la línea base con las búsquedas
# cv=5 usa StratifiedKFold, igual que las búsquedas con cv=5
cv = 5
cv_scores = cross_val_score(pipeline, X_train, y_train, cv=cv, scoring='recall', n_jobs=-1)

print('🔵 LÍNEA BASE — Validación cruzada (Recall) en train')
print(f'Recall por fold: {cv_scores}')
print(f'Recall medio:    {cv_scores.mean():.4f}')
print(f'Desv. estándar:  {cv_scores.std():.4f}')
print(f'Min:             {cv_scores.min():.4f}')
print(f'Max:             {cv_scores.max():.4f}')

🔵 LÍNEA BASE — Validación cruzada (Recall) en train
Recall por fold: [0.53846154 0.47157191 0.4548495  0.47491639 0.46153846]
Recall medio:    0.4803
Desv. estándar:  0.0300
Min:             0.4548
Max:             0.5385


---
# Fase 2: Búsqueda de hiperparámetros

**Objetivo:** Encontrar la mejor configuración de hiperparámetros para nuestro RandomForest, usando primero una búsqueda aleatoria amplia y luego un refinamiento preciso.

**Métrica de negocio:** Elegimos **Recall** como `scoring` porque StreamLoop pierde más dinero por no detectar a un cliente que cancela (Falso Negativo) que por ofrecer una retención innecesaria (Falso Positivo).

**Regla de oro:** Todo se hace con validación cruzada sobre el **train set**. El test set no se toca hasta el final.

In [15]:
# Espacio de búsqueda amplio para RandomForest
# Probamos valores variados para cada hiperparámetro
param_distributions = {
    'classifier__n_estimators': [50, 100, 200, 300, 500],
    'classifier__max_depth': [None, 5, 10, 15, 20, 30],
    'classifier__min_samples_split': [2, 5, 10, 20, 50],
    'classifier__min_samples_leaf': [1, 2, 5, 10, 20],
    'classifier__max_features': ['sqrt', 'log2', None],
    'classifier__bootstrap': [True, False]
}

print("Espacio de búsqueda amplio:")
for param, values in param_distributions.items():
    print(f"  {param}: {values}")
print(f"\nCombinaciones posibles: {np.prod([len(v) for v in param_distributions.values()]):,}")
print("RandomizedSearchCV probará 60 combinaciones al azar (frente a 4,500 posibles)")

Espacio de búsqueda amplio:
  classifier__n_estimators: [50, 100, 200, 300, 500]
  classifier__max_depth: [None, 5, 10, 15, 20, 30]
  classifier__min_samples_split: [2, 5, 10, 20, 50]
  classifier__min_samples_leaf: [1, 2, 5, 10, 20]
  classifier__max_features: ['sqrt', 'log2', None]
  classifier__bootstrap: [True, False]

Combinaciones posibles: 4,500
RandomizedSearchCV probará 60 combinaciones al azar (frente a 4,500 posibles)


### RandomizedSearchCV — Búsqueda aleatoria amplia

Usamos `RandomizedSearchCV` con:
- **n_iter=60**: probamos 60 combinaciones al azar de las 4.500 posibles
- **scoring='recall'**: porque el negocio prioriza detectar clientes que cancelan
- **cv=5**: validación cruzada de 5 folds
- **n_jobs=-1**: usamos todos los núcleos de la CPU
- **refit=True**: reentrena automáticamente el mejor modelo en todo el train set
- **random_state=42**: para que los resultados sean reproducibles

In [12]:
%%time
random_search = RandomizedSearchCV(
    pipeline,
    param_distributions=param_distributions,
    n_iter=60,
    cv=5,
    scoring='recall',
    n_jobs=-1,
    random_state=42,
    refit=True,      # reentrena el mejor modelo en todo el train set automáticamente
    verbose=1,
    return_train_score=False
)

random_search.fit(X_train, y_train)

print(f"\n✅ RandomizedSearchCV completado")
print(f"Mejor recall (CV): {random_search.best_score_:.4f}")
print(f"Mejores hiperparámetros: {random_search.best_params_}")

Fitting 5 folds for each of 60 candidates, totalling 300 fits

✅ RandomizedSearchCV completado
Mejor recall (CV): 0.5164
Mejores hiperparámetros: {'classifier__n_estimators': 50, 'classifier__min_samples_split': 2, 'classifier__min_samples_leaf': 1, 'classifier__max_features': None, 'classifier__max_depth': 10, 'classifier__bootstrap': False}
CPU times: user 4.35 s, sys: 293 ms, total: 4.65 s
Wall time: 6min 30s


### Análisis de resultados del RandomizedSearch

Revisamos `cv_results_` para ver:
1. El top-10 de combinaciones y su Recall promedio
2. La variación entre folds (desviación estándar)
3. Patrones: qué valores de hiperparámetros aparecen más en los mejores resultados

In [13]:
# Convertir cv_results_ a DataFrame para analizar
cv_results_df = pd.DataFrame(random_search.cv_results_)

# Columnas de interés
cols = ['mean_test_score', 'std_test_score', 'rank_test_score'] + \
       [c for c in cv_results_df.columns if c.startswith('param_')]

# Top 10 mejores combinaciones por Recall medio
top10 = cv_results_df.sort_values('rank_test_score').head(10)[cols]
print("🏆 Top 10 combinaciones (ordenadas por Recall medio):")
display(top10)

print(f"\n📊 Mejor Recall promedio: {random_search.best_score_:.4f}")
print(f"📊 Desviación estándar (mejor combo): {cv_results_df.loc[cv_results_df['rank_test_score']==1, 'std_test_score'].values[0]:.4f}")
print(f"📊 Rango de Recall entre top-10: {top10['mean_test_score'].min():.4f} - {top10['mean_test_score'].max():.4f}")

🏆 Top 10 combinaciones (ordenadas por Recall medio):


,mean_test_score,std_test_score,rank_test_score,param_classifier__n_estimators,param_classifier__min_samples_split,param_classifier__min_samples_leaf,param_classifier__max_features,param_classifier__max_depth,param_classifier__bootstrap
50,0.516388,0.011701,1,50,2,1,NaN,10,False
22,0.505017,0.025295,2,500,50,2,NaN,10,False
25,0.502341,0.023875,3,300,2,1,sqrt,15,True
55,0.496321,0.025785,4,200,2,1,NaN,None,True
45,0.496321,0.024992,5,100,5,2,log2,30,True
35,0.495652,0.027693,6,300,5,1,NaN,15,False
57,0.495652,0.025153,6,200,10,2,NaN,None,True
51,0.494314,0.027121,8,200,10,1,sqrt,10,False
1,0.493645,0.030842,9,100,10,5,NaN,15,False
53,0.493645,0.025348,9,300,2,5,log2,10,False



📊 Mejor Recall promedio: 0.5164
📊 Desviación estándar (mejor combo): 0.0117
📊 Rango de Recall entre top-10: 0.4936 - 0.5164


### GridSearchCV — Búsqueda precisa

El top-10 de `RandomizedSearchCV` sugiere una región prometedora: `max_depth` cerca de 10, `min_samples_leaf` 1–2 y pocos requisitos para dividir nodos. La mejor combinación usó 50 árboles, `max_features=None` y `bootstrap=False`; otras combinaciones competitivas probaron `sqrt` y `bootstrap=True`.

Por eso refinamos esos valores y mantenemos ambas alternativas de `max_features` y `bootstrap`, sin volver a recorrer todo el espacio amplio. La cuadrícula tendrá **96 combinaciones (480 ajustes con 5 folds)**.

| Hiperparámetro | Zona indicada por RandomizedSearch | Valores de GridSearch |
|---|---|---|
| `n_estimators` | 50 fue el mejor; también aparecen 100 en el top-10 | 50, 100 |
| `max_depth` | 10 fue el mejor; 15 también fue competitivo | 8, 10, 12 |
| `min_samples_split` | 2 fue el mejor; 5 aparece entre los primeros | 2, 5 |
| `min_samples_leaf` | 1 fue el mejor; 2 fue competitivo | 1, 2 |
| `max_features` | `None` fue el mejor; `sqrt` también aparece alto | `None`, `sqrt` |
| `bootstrap` | aparecen tanto `False` como `True` | `False`, `True` |

In [8]:
from math import prod
from sklearn.model_selection import GridSearchCV

# Espacio refinado a partir de las combinaciones competitivas del RandomizedSearch
param_grid = {
    'classifier__n_estimators': [50, 100],
    'classifier__max_depth': [8, 10, 12],
    'classifier__min_samples_split': [2, 5],
    'classifier__min_samples_leaf': [1, 2],
    'classifier__max_features': [None, 'sqrt'],
    'classifier__bootstrap': [False, True]
}

n_combinaciones = prod(len(v) for v in param_grid.values())
print(f"Combinaciones en GridSearch: {n_combinaciones:,}")
print(f"Total fits: {n_combinaciones} × 5 folds = {n_combinaciones * 5:,}")

Combinaciones en GridSearch: 96
Total fits: 96 × 5 folds = 480


### Ejecución de GridSearchCV

Corremos el GridSearch con el espacio acotado. Usamos `refit=True` para que reentrene el mejor modelo automáticamente.

In [9]:
%%time
grid_search = GridSearchCV(
    pipeline,
    param_grid=param_grid,
    cv=5,
    scoring='recall',
    n_jobs=-1,
    refit=True,
    verbose=1
)

grid_search.fit(X_train, y_train)

print(f"\n✅ GridSearchCV completado")
print(f"Mejor recall (CV): {grid_search.best_score_:.4f}")
print(f"Mejores hiperparámetros: {grid_search.best_params_}")

Fitting 5 folds for each of 96 candidates, totalling 480 fits



✅ GridSearchCV completado
Mejor recall (CV): 0.5231
Mejores hiperparámetros: {'classifier__bootstrap': False, 'classifier__max_depth': 8, 'classifier__max_features': None, 'classifier__min_samples_leaf': 2, 'classifier__min_samples_split': 2, 'classifier__n_estimators': 50}
CPU times: user 5.96 s, sys: 249 ms, total: 6.21 s
Wall time: 4min 4s


### Análisis de resultados del GridSearch

Revisamos el top-10, la variación entre folds y comparamos con la línea base.

In [14]:
# Resultados del GridSearch
grid_results = pd.DataFrame(grid_search.cv_results_)
cols_grid = ['mean_test_score', 'std_test_score', 'rank_test_score'] + \
            [c for c in grid_results.columns if c.startswith('param_')]

top10_grid = grid_results.sort_values('rank_test_score').head(10)[cols_grid]
print("🏆 Top 10 combinaciones (GridSearch):")
display(top10_grid)

print(f"\n📊 Mejor Recall promedio (GridSearch): {grid_search.best_score_:.4f}")
print(f"📊 Desviación estándar (mejor combo): {grid_results.loc[grid_results['rank_test_score']==1, 'std_test_score'].values[0]:.4f}")
print(f"📊 Rango de Recall entre top-10: {top10_grid['mean_test_score'].min():.4f} - {top10_grid['mean_test_score'].max():.4f}")

# Comparación línea base vs mejor modelo
print("\n" + "="*60)
print("📊 COMPARATIVA: Línea base vs Mejor modelo (validación cruzada)")
print("="*60)
print(f"Línea base    — Recall medio CV:    {cv_scores.mean():.4f} (±{cv_scores.std():.4f})")
print(f"Randomized    — Mejor Recall CV:    {random_search.best_score_:.4f}")
print(f"GridSearch    — Mejor Recall CV:    {grid_search.best_score_:.4f}")
print(f"\nMejora total del Recall: {(grid_search.best_score_ - cv_scores.mean()) * 100:.2f} puntos porcentuales")

🏆 Top 10 combinaciones (GridSearch):


,mean_test_score,std_test_score,rank_test_score,param_classifier__bootstrap,param_classifier__max_depth,param_classifier__max_features,param_classifier__min_samples_leaf,param_classifier__min_samples_split,param_classifier__n_estimators
5,0.523077,0.057174,1,False,8,NaN,2,2,100
4,0.523077,0.057174,1,False,8,NaN,2,2,50
7,0.520401,0.057369,3,False,8,NaN,2,5,100
6,0.520401,0.057369,3,False,8,NaN,2,5,50
1,0.519064,0.057401,5,False,8,NaN,1,2,100
0,0.519064,0.057362,5,False,8,NaN,1,2,50
17,0.518395,0.012691,7,False,10,NaN,1,2,100
3,0.517726,0.057009,8,False,8,NaN,1,5,100
2,0.517057,0.058182,9,False,8,NaN,1,5,50
16,0.516388,0.011701,10,False,10,NaN,1,2,50



📊 Mejor Recall promedio (GridSearch): 0.5231
📊 Desviación estándar (mejor combo): 0.0572
📊 Rango de Recall entre top-10: 0.5164 - 0.5231

📊 COMPARATIVA: Línea base vs Mejor modelo (validación cruzada)
Línea base    — Recall medio CV:    0.4803 (±0.0300)
Randomized    — Mejor Recall CV:    0.5164
GridSearch    — Mejor Recall CV:    0.5231

Mejora total del Recall: 4.28 puntos porcentuales


---
# Fase 3: Selección del modelo final

**Objetivo:** Elegir el modelo con mejor equilibrio entre Recall promedio y estabilidad entre folds, justificar la decisión, y evaluarlo en el test set exactamente una vez.

### 1. Análisis de cv_results_ — Trade-off entre promedio y estabilidad

No nos quedamos con el que tiene el mejor promedio sin más. Revisamos también cuánto varía entre folds (std_test_score). Un modelo con promedio ligeramente menor pero mucha menos varianza puede ser más fiable en producción.

In [16]:
# Analizar el trade-off entre Recall medio y estabilidad en los candidatos del GridSearch
# Buscamos combinaciones con buena relación recall / std

grid_analysis = grid_results[cols_grid].copy()
grid_analysis['recall_menos_std'] = grid_analysis['mean_test_score'] - grid_analysis['std_test_score']
grid_analysis['coef_variacion'] = (grid_analysis['std_test_score'] / grid_analysis['mean_test_score']) * 100

# Top 5 por mejor Recall medio
print("🏆 Top 5 por Recall medio (con estabilidad):")
display(grid_analysis.sort_values('mean_test_score', ascending=False).head(5))

# Top 5 por mejor recall_menos_std (Recall - desviación, penaliza inestabilidad)
print("\n⚖️  Top 5 por Recall - std (penaliza varianza alta):")
display(grid_analysis.sort_values('recall_menos_std', ascending=False).head(5))

print("\n🔍 Observaciones:")
print(f"  - Mejor promedio:    {grid_analysis.iloc[grid_analysis['mean_test_score'].idxmax()]['mean_test_score']:.4f}")
print(f"    pero std:          {grid_analysis.iloc[grid_analysis['mean_test_score'].idxmax()]['std_test_score']:.4f}")
print(f"  - Mejor Recall-std:  {grid_analysis.iloc[grid_analysis['recall_menos_std'].idxmax()]['mean_test_score']:.4f}")
print(f"    con std:           {grid_analysis.iloc[grid_analysis['recall_menos_std'].idxmax()]['std_test_score']:.4f}")

🏆 Top 5 por Recall medio (con estabilidad):


,mean_test_score,std_test_score,rank_test_score,param_classifier__bootstrap,param_classifier__max_depth,param_classifier__max_features,param_classifier__min_samples_leaf,param_classifier__min_samples_split,param_classifier__n_estimators,recall_menos_std,coef_variacion
5,0.523077,0.057174,1,False,8,NaN,2,2,100,0.465903,10.930325
4,0.523077,0.057174,1,False,8,NaN,2,2,50,0.465903,10.930325
7,0.520401,0.057369,3,False,8,NaN,2,5,100,0.463032,11.024052
6,0.520401,0.057369,3,False,8,NaN,2,5,50,0.463032,11.024052
1,0.519064,0.057401,5,False,8,NaN,1,2,100,0.461663,11.058473



⚖️  Top 5 por Recall - std (penaliza varianza alta):


,mean_test_score,std_test_score,rank_test_score,param_classifier__bootstrap,param_classifier__max_depth,param_classifier__max_features,param_classifier__min_samples_leaf,param_classifier__min_samples_split,param_classifier__n_estimators,recall_menos_std,coef_variacion
17,0.518395,0.012691,7,False,10,NaN,1,2,100,0.505703,2.448215
16,0.516388,0.011701,10,False,10,NaN,1,2,50,0.504687,2.265914
18,0.511037,0.007449,14,False,10,NaN,1,5,50,0.503588,1.457530
19,0.511037,0.007743,14,False,10,NaN,1,5,100,0.503294,1.515162
21,0.511706,0.011391,12,False,10,NaN,2,2,100,0.500315,2.226064



🔍 Observaciones:
  - Mejor promedio:    0.5231
    pero std:          0.0572
  - Mejor Recall-std:  0.5184
    con std:           0.0127


### 2. Elección del modelo final

Basándonos en el análisis anterior:

| Candidato | Recall medio | std | Coef. variación |
|---|---|---|---|
| **A** — depth=8, leaf=2, split=2 → máximo promedio | 0.5231 | 0.0572 | 10.9% |
| **B ← ELEGIDO** — depth=10, leaf=1, split=2, n=100 → balance | **0.5184** | **0.0127** | **2.4%** |
| **C** — depth=10, leaf=1, split=5, n=50 → máxima estabilidad | 0.5110 | 0.0074 | 1.5% |

**Decisión:** Elegimos **Candidato B** (`max_depth=10, min_samples_leaf=1, min_samples_split=2, max_features=None, bootstrap=False, n_estimators=100`).

**Justificación:** El candidato A tiene el mejor promedio (0.5231) pero una desviación entre folds 4.5× mayor (0.0572 vs 0.0127). En producción esto significa que el modelo podría rendir mucho peor en ciertos batches de clientes. El candidato B sacrifica solo **0.005 puntos** de Recall promedio a cambio de ser **mucho más consistente** entre folds. El candidato C es muy estable pero pierde 0.007 más de Recall que B, que no vale la pena cuando B ya tiene buena estabilidad (2.4%).

Para el negocio: un modelo que consistentemente detecta ~52% de los churners (con poca variación) permite planificar mejor las campañas de retención que un modelo que a veces detecta 58% y a veces 45%.

In [17]:
# Extraer los mejores hiperparámetros del candidato B (balance recall-estabilidad)
best_params_balanced = {
    'classifier__n_estimators': 100,
    'classifier__max_depth': 10,
    'classifier__min_samples_split': 2,
    'classifier__min_samples_leaf': 1,
    'classifier__max_features': None,
    'classifier__bootstrap': False
}

# Construir pipeline con estos hiperparámetros
pipeline_final = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(
        n_estimators=best_params_balanced['classifier__n_estimators'],
        max_depth=best_params_balanced['classifier__max_depth'],
        min_samples_split=best_params_balanced['classifier__min_samples_split'],
        min_samples_leaf=best_params_balanced['classifier__min_samples_leaf'],
        max_features=best_params_balanced['classifier__max_features'],
        bootstrap=best_params_balanced['classifier__bootstrap'],
        random_state=42
    ))
])

# Entrenar en train completo
pipeline_final.fit(X_train, y_train)

print('✅ Modelo final entrenado (candidato B — balance recall/estabilidad)')
print(f'Hiperparámetros: {best_params_balanced}')

✅ Modelo final entrenado (candidato B — balance recall/estabilidad)
Hiperparámetros: {'classifier__n_estimators': 100, 'classifier__max_depth': 10, 'classifier__min_samples_split': 2, 'classifier__min_samples_leaf': 1, 'classifier__max_features': None, 'classifier__bootstrap': False}


### 3. Evaluación final en test set

**⚠️ Única y última vez que tocamos el test set.** Usamos las mismas métricas que en la línea base para comparación directa.

In [20]:
# Evaluar modelo final en test
y_pred_final = pipeline_final.predict(X_test)
y_pred_proba_final = pipeline_final.predict_proba(X_test)[:, 1]

# Mismas métricas que la línea base
accuracy_final = accuracy_score(y_test, y_pred_final)
precision_final = precision_score(y_test, y_pred_final)
recall_final = recall_score(y_test, y_pred_final)
f1_final = f1_score(y_test, y_pred_final)
roc_auc_final = roc_auc_score(y_test, y_pred_final)

print("=" * 60)
print("🔵 LÍNEA BASE vs 🟢 MODELO FINAL — Comparación en test set")
print("=" * 60)
print(f"{'Métrica':<15} {'Línea base':<12} {'Modelo final':<12} {'Cambio':<10}")
print("-" * 50)
print(f"{'Accuracy':<15} {accuracy:<12.4f} {accuracy_final:<12.4f} {(accuracy_final - accuracy):<+10.4f}")
print(f"{'Precision':<15} {precision:<12.4f} {precision_final:<12.4f} {(precision_final - precision):<+10.4f}")
print(f"{'Recall':<15} {recall:<12.4f} {recall_final:<12.4f} {(recall_final - recall):<+10.4f}")
print(f"{'F1-score':<15} {f1:<12.4f} {f1_final:<12.4f} {(f1_final - f1):<+10.4f}")
print(f"{'ROC AUC':<15} {roc_auc:<12.4f} {roc_auc_final:<12.4f} {(roc_auc_final - roc_auc):<+10.4f}")

print(f"\n📊 Matriz de confusión — Modelo final:")
cm_final = confusion_matrix(y_test, y_pred_final)
print(cm_final)
print(f"\n  VP (detectó churn real):  {cm_final[1,1]}")
print(f"  FN (churn no detectado):  {cm_final[1,0]}")
print(f"  FP (falsa alarma):        {cm_final[0,1]}")
print(f"  VN (correcto no churn):   {cm_final[0,0]}")

print(f"\n📊 Classification Report — Modelo final:")
print(classification_report(y_test, y_pred_final))

🔵 LÍNEA BASE vs 🟢 MODELO FINAL — Comparación en test set
Métrica         Línea base   Modelo final Cambio    
--------------------------------------------------
Accuracy        0.7921       0.7580       -0.0341   
Precision       0.6421       0.5455       -0.0967   
Recall          0.4893       0.5294       +0.0401   
F1-score        0.5554       0.5373       -0.0181   
ROC AUC         0.6954       0.6850       -0.0104   

📊 Matriz de confusión — Modelo final:
[[870 165]
 [176 198]]

  VP (detectó churn real):  198
  FN (churn no detectado):  176
  FP (falsa alarma):        165
  VN (correcto no churn):   870

📊 Classification Report — Modelo final:
              precision    recall  f1-score   support

           0       0.83      0.84      0.84      1035
           1       0.55      0.53      0.54       374

    accuracy                           0.76      1409
   macro avg       0.69      0.68      0.69      1409
weighted avg       0.76      0.76      0.76      1409



---
## Conclusión del proyecto

### Resumen del flujo de trabajo

| Fase | Qué se hizo |
|---|---|
| **1. Línea base** | Pipeline + RandomForest por defecto → **Recall 0.4893** |
| **2a. RandomizedSearchCV** | 60 combinaciones sobre 4.500 posibles → mejor zona identificada |
| **2b. GridSearchCV** | 96 combinaciones alrededor de la zona prometedora → refinamiento |
| **3. Selección final** | Trade-off promedio vs estabilidad → modelo elegido (B) |
| **Evaluación final** | Una sola evaluación en test → **Recall 0.5294** (+4.01 pp) |

La mejora de **+4 puntos de Recall** se logró priorizando un modelo consistente entre folds (coeficiente de variación del 2.4% frente al 10.9% del modelo con mejor promedio crudo). Para StreamLoop, esto significa poder planificar campañas de retención con una expectativa fiable de cuántos churners se van a detectar.